# Ring-attractor student — phase portraits & perturbation

Streamplot + trajectory phase portraits with fixed points, perturbation snapshots, and across-model Pearson *r* summary.

In [ ]:
import pickle
import socket
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

np.random.seed(0)
torch.manual_seed(0)

sys.path.insert(0, str(Path.cwd().parent))

from fig_utils.fixed_points import get_multi_scale_jitter
from fig_utils.perturbation import (
    compute_perturbation_distance_stats,
    generate_w_perturb_x,
    perturbation_goal_bounds,
)
from fig_utils.plots import (
    compute_ring_subspace_flow,
    plot_boxplot_by_group,
    plot_fp_counts_barplot,
    plot_perturbation_latent_snapshots_attractor,
    plot_ring_demo_trajectories,
    plot_ring_subspace_streamplot,
    subplots_panels,
)
from fig_utils.transformed_rnn import transformed_rnn
from vi_rnn.data_utils import make_all_trials, stim_end_bins
from vi_rnn.datasets import TS_dataset_multi
from vi_rnn.fixed_points import run_scify
from vi_rnn.generate import generate
from vi_rnn.saving import load_model, CPU_Unpickler
from vi_rnn.utils import get_orth_proj_latents

%matplotlib inline

In [ ]:
cmap = mpl.colors.ListedColormap(sns.color_palette("husl", n_colors=6))

hostname = socket.gethostname()
print("hostname:", hostname)

if hostname == "MatthijsDesktop":
    out_dir = Path("/home/matthijs/swm_rnn/final_models/synthetic")
    data_root = Path("/home/matthijs/swm_rnn/data")
else:
    out_dir = Path("/Users/matthijs/swm_rnn/final_models/synthetic")
    data_root = Path.cwd().parent / "data"
# seed = 172830
# https://github.com/mackelab/swm_rnn/blob/d662758afa3c3432bdeafc341b7d9d68683821db/generate_figures/09_Supp_ring_attractor.ipynb

model_dirs = [
    # "ring_attractor_low_rank_one_to_one_dim_z_2_date_2026_06_06_T_20_10_03",
    # "ring_attractor_low_rank_one_to_one_dim_z_2_date_2026_06_06_T_18_34_57",
    "ring_attractor_low_rank_one_to_one_dim_z_2_date_2026_06_04_T_21_13_55",
    "ring_attractor_low_rank_one_to_one_dim_z_2_date_2026_06_04_T_01_33_14",
    "ring_attractor_low_rank_one_to_one_dim_z_2_date_2026_06_04_T_15_38_49",
]
# model_dirs = ["ring_attractor_low_rank_one_to_one_dim_z_2_date_2026_06_12_T_13_29_00"]

In [ ]:
# --- controls ---
P = 6
n_repeats_pert = 100
noise_scale = 1.0
noise_scale_demo = 1.0
bin_size = 0.05

generate_plots = True
run = True

In [ ]:
_task_pkl = Path(str(out_dir / model_dirs[0]) + "_task_params.pkl")
with open(_task_pkl, "rb") as f:
    task_params = CPU_Unpickler(f).load()

data_prefix = Path(task_params["path"])
seed_st = int(task_params["seed"])
if not Path(f"{data_prefix}_y_seed_{seed_st}.npy").exists():
    raise FileNotFoundError(
        f"Dataset not found at {data_prefix} (expected {data_prefix}_y_seed_{seed_st}.npy)."
    )

task = TS_dataset_multi(task_params)
dt = float(task_params["bin_size"])

u_pert_master, _, _, _ = make_all_trials(
    task_params,
    dur=8,
    n_stim=P,
    n_pos=1,
    cue_dur=-1,
    bin_size=bin_size,
    interval_dur="mean",
    delay_dur="mean",
)

In [ ]:
def trial_indices_by_condition(labels, n_conds, n_repeats):
    inds = []
    for p in range(n_conds):
        pool = np.where(labels == p)[0]
        if len(pool) < n_repeats:
            raise ValueError(
                f"condition {p}: need {n_repeats} trials, have {len(pool)}"
            )
        inds.extend(np.random.choice(pool, size=n_repeats, replace=False))
    return np.array(inds, dtype=int)

In [ ]:
if run:
    rows = []

    for model_name in model_dirs:
        model_path = out_dir / model_name
        print("Loading", model_path)

        vae, training_params, task_params_m = load_model(
            str(model_path), load_encoder=False, backward_compat=False
        )
        print(
            "dim_z:",
            vae.dim_z,
            "| centroid_loss:",
            training_params.get("centroid_loss_weight"),
        )

        projection_matrix = get_orth_proj_latents(vae).cpu().numpy()
        rnn_proj = transformed_rnn(vae, projection_matrix, np.zeros(vae.dim_z))

        # --- latents for phase portraits (noiseless, extended input) ---
        u_fp = task.sessions[0].stim[:50]
        u_fp = torch.cat([u_fp, *[torch.zeros_like(u_fp)] * 10], dim=2)
        Zo, _, _, _ = generate(
            vae, u=u_fp, k=1, noise_scale=0.0, initial_state="prior_sample"
        )
        Zo_np = Zo.detach().cpu().numpy()
        if Zo_np.ndim == 3:
            Zo_np = Zo_np[..., None]
        Zt = np.einsum("ij,bjtk->kbit", projection_matrix, Zo_np)

        k1_tr = Zt[0, 2:, 0, 2:]
        k2_tr = Zt[0, 2:, 1, 2:]
        if k1_tr.size == 0:
            k1_tr, k2_tr = Zt[0, :, 0, :], Zt[0, :, 1, :]
        pad = 0.12
        xlim = (k1_tr.min() - pad * np.ptp(k1_tr), k1_tr.max() + pad * np.ptp(k1_tr))
        ylim = (k2_tr.min() - pad * np.ptp(k2_tr), k2_tr.max() + pad * np.ptp(k2_tr))

        # --- fixed points (SCY-FI) ---
        v0 = np.zeros(rnn_proj.dim_u)
        I, h2, W2_full = rnn_proj.pI, rnn_proj.h2, rnn_proj.W2
        A_fp, W1, h1 = rnn_proj.tau, rnn_proj.W1, rnn_proj.h1

        Z_flat = Zt[0].reshape(-1, rnn_proj.dim_z)
        x = Z_flat @ W2_full.T + h2 + I @ v0
        D_init = (x > 0).astype("uint8")
        Z_end = np.column_stack([k1_tr[:, -1], k2_tr[:, -1]])
        D_end = (Z_end @ W2_full.T + h2 + I @ v0 > 0).astype("uint8")
        unique_patterns = get_multi_scale_jitter(
            Z_flat,
            W2_full=W2_full,
            h2=h2,
            I=I,
            v=v0,
            scales=(0.1, 0.5, 2.0, 5.0),
            samples_per_scale=100,
        )
        D_combined = np.unique(np.vstack([D_init, D_end, unique_patterns]), axis=0)
        print(f"SCY-FI patterns: {len(D_combined)}")

        found_lower_orders, found_eigvals, n_inverses = run_scify(
            A=A_fp,
            W1=W1,
            W2=W2_full,
            h1=h1,
            h2=h2,
            order=1,
            inner_loop_iterations=10,
            round_dec=6,
            n_inverses_max=100_000,
            initial_states=D_combined,
        )
        evs = np.array(found_eigvals[0])
        Z_fps = np.array(found_lower_orders[0])[:, 0]
        max_eig_mags = [np.max(np.abs(e)) for e in evs]
        fp_stable = np.array(max_eig_mags) < 1
        fp_unstable = np.array([sum(np.abs(e) > 1) == 2 for e in evs])
        fp_saddle = ~fp_stable & ~fp_unstable
        n_stable_fps = int(fp_stable.sum())
        n_unstable_fps = int(fp_unstable.sum())
        n_saddle_fps = int(fp_saddle.sum())
        print(
            f"Fixed points: {Z_fps.shape[0]} | stable: {n_stable_fps} | "
            f"unstable: {n_unstable_fps} | saddles: {n_saddle_fps} | "
            f"inverses: {n_inverses[0]}"
        )

        zsp1, zsp2, u_flow, v_flow = compute_ring_subspace_flow(
            rnn_proj,
            xlim=xlim,
            ylim=ylim,
        )

        if generate_plots:
            fig, ax = plot_ring_subspace_streamplot(
                zsp1,
                zsp2,
                u_flow,
                v_flow,
                Z_fps,
                evs,
                cmap=cmap,
                plot_saddles=False,
                xlabel=r"$z_1$",
                ylabel=r"$z_2$",
                show=False,
            )
            # for i in range(k1_tr.shape[0]):
            #    ax.plot(k1_tr[i], k2_tr[i], color="0.35", alpha=0.5, lw=0.8, zorder=5)
            ax.set_xlim(xlim)
            ax.set_ylim(ylim)
            plt.tight_layout()
            plt.show()
            plt.close(fig)

            n_demo = 12
            u_demo = task.sessions[0].stim[:n_demo]
            u_demo = torch.cat([u_demo, torch.zeros_like(u_demo[:, :, :50])], dim=2)
            Z0 = np.random.randn(n_demo, vae.dim_z) * noise_scale_demo
            rnn_proj.z0 = Z0
            Z_hist = rnn_proj.simulate(
                u_demo.detach().cpu().numpy(), noise_scale=noise_scale_demo
            )
            plot_ring_demo_trajectories(
                Z_hist,
                Z_fps,
                evs,
                zsp1,
                zsp2,
                cmap=cmap,
                panel_gap_x=0.5,
                show=True,
            )

        rows.append(
            {
                "name": model_name,
                "path": str(model_path),
                "dim_z": vae.dim_z,
                "centroid_loss": training_params.get("centroid_loss_weight"),
                "macaque": f"dz{vae.dim_z}",
                "n_stable_fps": n_stable_fps,
                "n_unstable_fps": n_unstable_fps,
                "n_saddle_fps": n_saddle_fps,
            }
        )

In [ ]:
if run:
    df = pd.DataFrame(rows)
    pickle.dump(df, open("../data/processed/df_perturb_ring_student.pkl", "wb"))
else:
    df = pickle.load(open("../data/processed/df_perturb_ring_student.pkl", "rb"))

In [ ]:
# Summary: perturbation r (notebook 05 style) + fixed-point counts by type
order = [str(d) for d in sorted(df["dim_z"].unique())]
plot_df = df.copy()
plot_df["dim_z_str"] = plot_df["dim_z"].astype(str)


plot_fp_counts_barplot(
    df,
    teacher_stable=6,
    teacher_unstable=1,
    teacher_saddle=6,
    # ax=axes[1],
    box_w=1.5,
    show=False,
)
plt.show()

df